# Stellar Coordinate Explorer - Adding Galactic Coordinates (Biased)

## Objective
Transform the ICRS (RA, Dec) coordinates of every star in your filtered table to Galactic coordinates (l, b) and store them as new columns.

## Why this matters
- Galactic coordinates reveal how stars are distributed or structured relative to the Milky Way's plane in way that is not obvious in equitorial coordinates.
- Later visualizations (Aitoff projection) will use Galactic coordinates to show the whole sky.

## Dataset
- Input `bright_stars_filtered.fits` (saved from Day 3)
- Columns: `source_id`, `ra`, `dec`, `parallax`, `phot_g_mean_mag`, `bp_rp`
- Output: `stars_with_galactic_coord_biased.fits`

## Goals for Today
- Load the 10000 sources table
- Create a `SkyCoord` object for all rows using RA and Dec (in degrees)
- Transform to Galactic Coordinates
- Add two new columns: `gal_l` (longitude) and `gal_b` (latitude)
- Verify that the transformed coordinates are within expected ranges ($l:\ 0-360^\degree$, $b:\ -90^\degree$ to $+90^\degree$)
- Save the enhanced table as `stars_with_galactic_biased.fits`

## Checkpoint
- Table loaded successfully
- SkyCoord created without errors (watch for unit issues)
- New columns added and contain sensible numbers
- Quick Verification: print first 10 rows showing `RA`, `Dec`, `gal_l`, `gal_b`
- Enhanced table saved to disk

## Code

In [ ]:
# Setup
from astropy.coordinates import SkyCoord, Galactic
from astropy.table import Table
import astropy.units as u


Now let's load the 10,000 sources to a Table 

In [ ]:
table = Table.read("../../data/gaia_subset_biased.fits")
table.info()

<Table length=10000>
      name       dtype  unit    class     n_bad
--------------- ------- ---- ------------ -----
      source_id   int64            Column     0
             ra float64  deg       Column     0
            dec float64  deg       Column     0
       parallax float64  mas       Column     0
phot_g_mean_mag float32  mag       Column     0
          bp_rp float32  mag MaskedColumn    14


### Obtaining the Equitorial Coordinates and Transforming to Galactic
Let's create a SkyCoord object containing equitorial frame coordinates for all the sources.

In [ ]:
eq_coord = SkyCoord(table["ra"], table["dec"], unit=(u.deg, u.deg), frame="icrs")

Confirming that the coordinates were saved

In [ ]:
print(f"{'#' * 20} Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame {'#' * 20}")
print(eq_coord)
eq_coord.ra, eq_coord.dec

#################### Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame ####################
<SkyCoord (ICRS): (ra, dec) in deg
    [( 45.15296534,  0.38634212), ( 44.2634464 ,  1.50159246),
     ( 48.30793452,  3.82011414), ..., (129.92755274, 19.778397  ),
     (130.45849394, 19.87411206), (130.47124403, 20.15937977)]>


(<Longitude [ 45.15296534,  44.2634464 ,  48.30793452, ..., 129.92755274,
             130.45849394, 130.47124403] deg>,
 <Latitude [ 0.38634212,  1.50159246,  3.82011414, ..., 19.778397  ,
            19.87411206, 20.15937977] deg>)

Now let's transform these coordinates to the Galactic frame.

In [ ]:
gal_coord = eq_coord.transform_to(Galactic())
print(f"{'#' * 20} Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame {'#' * 20}")
print(gal_coord)
gal_coord.b, gal_coord.l

#################### Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame ####################
<SkyCoord (Galactic): (l, b) in deg
    [(176.70133826, -48.52417579), (174.56403149, -48.36635708),
     (176.32146321, -43.87022615), ..., (205.72879396,  32.37066029),
     (205.82655171,  32.87250896), (205.51209975,  32.98139929)]>


(<Latitude [-48.52417579, -48.36635708, -43.87022615, ...,  32.37066029,
             32.87250896,  32.98139929] deg>,
 <Longitude [176.70133826, 174.56403149, 176.32146321, ..., 205.72879396,
             205.82655171, 205.51209975] deg>)

Now we can confirm whether these coordinates are within the expected ranges in the Galactic frame ($l:\ 0-360^\degree$, $b:\ -90^\degree$ to $+90^\degree$).

In [ ]:
print(f"l range: {gal_coord.l.min()} to {gal_coord.l.max()}")
print(f"b range: {gal_coord.b.min()} to {gal_coord.b.max()}")

l range: 113.31385709407242 deg to 228.11729603512464 deg
b range: -49.012328676412665 deg to 71.00558468928028 deg


Let's now save the new Galactic Coordinates in our table.

In [ ]:
table.add_column(gal_coord.l, name='gal_l', index=3)
table.add_column(gal_coord.b, name='gal_b', index=4)
print(f"{'#' * 20} Confirm the Galactic Coordinates for the Gaia stars {'#' * 20}")
print(table.info())


#################### Confirm the Galactic Coordinates for the Gaia stars ####################
<Table length=10000>
      name       dtype  unit    class     n_bad
--------------- ------- ---- ------------ -----
      source_id   int64            Column     0
             ra float64  deg       Column     0
            dec float64  deg       Column     0
          gal_l float64  deg       Column     0
          gal_b float64  deg       Column     0
       parallax float64  mas       Column     0
phot_g_mean_mag float32  mag       Column     0
          bp_rp float32  mag MaskedColumn    14
None


In [ ]:
print(f"{'#' * 20} View the first 10 sources {'#' * 20}")
table[:10]

#################### View the first 10 sources ####################


source_id,ra,dec,gal_l,gal_b,parallax,phot_g_mean_mag,bp_rp
,deg,deg,deg,deg,mas,mag,mag
int64,float64,float64,float64,float64,float64,float32,float32
16733192740608,45.152965337754885,0.3863421150872851,176.7013382587317,-48.52417578975809,5.473905596643335,9.925234,0.7235527
407785670617600,44.263446403533244,1.5015924603581685,174.56403149246566,-48.36635707636529,5.327594570736073,9.749034,0.5830326
2517233088211712,48.307934519908706,3.82011414311565,176.32146321483845,-43.87022615174943,16.796145185469527,9.589524,1.2472496
2710712774989440,48.1134802323883,3.964476089897251,175.99000095630257,-43.90784619837817,8.89349519082703,9.847445,0.8104763
3504422731187584,46.25523412225751,4.465545045836133,173.6513701247457,-44.85237855948805,5.850867569879387,9.01819,1.1034575
5509107306622592,43.55217071489401,5.048257834044321,170.249304551969,-46.23834713059198,5.53089203131021,7.6163025,0.19230413
5922077001986176,40.340090955371934,4.427539209105315,167.1967481104543,-48.76484822079696,6.990293076484558,6.9498754,0.3729596
9739615733526016,52.84779612797096,6.981660493359696,177.43389150882663,-38.418832620273115,5.469212664603673,9.330736,0.7043781


Now we save our table to a file named `stars_with_galactic_biased.fits`.

In [ ]:
table.write('../../data/stars_with_galactic_coord_biased.fits')

## Basic Observations

The range in galactic coordinates is approximately 113 to 228 in the galactic longitude and approximately -49 to 71 in the galactic latitude. This indicates that the sources are not widely distributed across the sky which is expected for our biased selection from the Gaia archive.